# EDA подготовленного ToolACE

Смотрим те же файлы, которые идут в GRPO: `data/experiments/`. Каждый ход ассистента в принятом диалоге становится отдельным примером. В промпт входит история до этого хода, эталон — сам ход, инструменты — список из system.

Ноутбук только читает уже подготовленные jsonl и `preparation.json`. Повторный разбор сырого ToolACE здесь не запускается.

In [1]:
import json
from collections import Counter
from pathlib import Path

from prepare import load_examples, render_prompt
from train import load_tokenizer

DATA = Path("data/experiments")
report = json.loads((DATA / "preparation.json").read_text())
train = load_examples(DATA / "train.jsonl")
validation = load_examples(DATA / "validation.jsonl")
print("train", len(train), "validation", len(validation))
print("исходных диалогов", report["source_dialogues"], "seed", report["seed"])

train 11824 validation 1292
исходных диалогов 11300 seed 42


## Что не попало в обучение

`prepare.py` выбрасывает диалог целиком. Причины в `preparation.json` сырые: у ошибок разбора там ещё и адрес объекта. Ниже они сгруппированы по типу сбоя.

In [2]:
def exclusion_bucket(reason):
    if reason.startswith("Пустая реплика"):
        return "пустая реплика ассистента"
    if reason.startswith("Не найдено начало"):
        return "в system нет списка инструментов"
    if "assignment" in reason:
        return "в аргументе присваивание, не литерал"
    if reason.startswith("invalid syntax"):
        return "вызов не разбирается как выражение"
    if reason.startswith("malformed"):
        return "аргумент не является литералом Python"
    return reason


buckets = Counter(exclusion_bucket(item["reason"]) for item in report["excluded_dialogues"])
kept_dialogues = len(report["train_dialogue_ids"]) + len(report["validation_dialogue_ids"])
print(f"принято диалогов {kept_dialogues} из {report['source_dialogues']}")
print(f"выброшено диалогов {len(report['excluded_dialogues'])}")
for name, count in buckets.most_common():
    print(f"  {count:4d}  {name}")

принято диалогов 10641 из 11300
выброшено диалогов 659
   518  в system нет списка инструментов
    96  аргумент не является литералом Python
    31  в аргументе присваивание, не литерал
     9  вызов не разбирается как выражение
     5  пустая реплика ассистента


## Как устроен один пример

Считаем историю до эталона, номер хода ассистента, вызовы и типы аргументов. Номер хода 0 — модель отвечает на первый вопрос. Ход 1 и дальше уже видит предыдущие вызовы и ответы инструментов.

In [ ]:
def value_type(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, int):
        return "int"
    if isinstance(value, float):
        return "float"
    if isinstance(value, str):
        return "str"
    if isinstance(value, list):
        return "list"
    if isinstance(value, dict):
        return "object"
    return type(value).__name__


def describe(messages, answer, tools):
    calls = [item["function"] for item in answer.get("tool_calls", [])]
    argument_types = []
    nested_arguments = 0
    for call in calls:
        for value in call["arguments"].values():
            argument_types.append(value_type(value))
            if isinstance(value, (list, dict)):
                nested_arguments += 1
    return {
        "tools": len(tools),
        "history": len(messages),
        "assistant_turn": sum(message["role"] == "assistant" for message in messages),
        "saw_tool_result": any(message["role"] == "tool" for message in messages),
        "calls": len(calls),
        "text_answer": not calls,
        "argument_types": argument_types,
        "nested_arguments": nested_arguments,
        "called_names": [call["name"] for call in calls],
        "available_names": [tool["function"]["name"] for tool in tools],
    }


rows = [describe(*example) for example in train + validation]
print("примеров", len(rows))

In [ ]:
def show_counts(title, counter, total):
    print(title)
    for key, count in sorted(counter.items(), key=lambda item: (-item[1], str(item[0]))):
        print(f"  {str(key):>8}  {count:6d}  {count / total:6.1%}")


total = len(rows)
text_answers = sum(row["text_answer"] for row in rows)
later_turns = sum(row["assistant_turn"] > 0 for row in rows)
with_tool_result = sum(row["saw_tool_result"] for row in rows)
nested = sum(row["nested_arguments"] > 0 for row in rows)
parallel = sum(row["calls"] >= 2 for row in rows)
print(f"ответ текстом, без вызова     {text_answers:6d}  {text_answers / total:6.1%}")
print(f"эталон не на первом ходе      {later_turns:6d}  {later_turns / total:6.1%}")
print(f"в истории уже есть tool       {with_tool_result:6d}  {with_tool_result / total:6.1%}")
print(f"два и больше вызова           {parallel:6d}  {parallel / total:6.1%}")
print(f"есть вложенный list/object    {nested:6d}  {nested / total:6.1%}")
print()
show_counts("вызовов в эталоне", Counter(row["calls"] for row in rows), total)
print()
show_counts("номер хода ассистента", Counter(row["assistant_turn"] for row in rows), total)
print()
show_counts("инструментов в prompt", Counter(row["tools"] for row in rows), total)
print()
argument_types = Counter(kind for row in rows for kind in row["argument_types"])
show_counts("типы значений аргументов", argument_types, sum(argument_types.values()))

ответ текстом, без вызова       3307   25.2%
эталон не на первом ходе        2475   18.9%
в истории уже есть tool         2408   18.4%
два и больше вызова             4061   31.0%
есть вложенный list/object      1918   14.6%

вызовов в эталоне
         1    5748   43.8%
         0    3307   25.2%
         2    1660   12.7%
         3    1211    9.2%
         4     907    6.9%
         5     195    1.5%
         6      63    0.5%
         7      17    0.1%
         8       5    0.0%
         9       2    0.0%
        12       1    0.0%

номер хода ассистента
         0   10641   81.1%
         1     784    6.0%
         2     778    5.9%
         3     524    4.0%
         4     297    2.3%
         5      92    0.7%

инструментов в prompt
         1    2731   20.8%
         2    2256   17.2%
         3    2242   17.1%
         4    1809   13.8%
         5    1737   13.2%
         6    1618   12.3%
         7     256    2.0%
         0     248    1.9%
         8     191    1.5%
        

## Имена функций

Смотрим, повторяются ли имена между примерами. Если почти каждое имя встречается один раз, модель не может выучить закрытый каталог функций и должна читать схему из промпта.

In [ ]:
available = Counter(name for row in rows for name in row["available_names"])
called = Counter(name for row in rows for name in row["called_names"])
once = sum(count == 1 for count in available.values())
print(f"уникальных имён в схемах   {len(available)}")
print(f"из них встречаются 1 раз    {once}  ({once / len(available):.1%})")
print(f"уникальных имён в эталонах  {len(called)}")
print("чаще всего вызваны:")
for name, count in called.most_common(8):
    print(f"  {count:4d}  {name}")

уникальных имён в схемах   15926
из них встречаются 1 раз    6960  (43.7%)
уникальных имён в эталонах  10448
чаще всего вызваны:
    14  Search
    13  getInventory
    10  Get Order
    10  Get Product
    10  getUVIndex
     9  DownloadFile
     9  getUserByName
     9  getOrderById


## Длина в токенах Qwen

Промпт рендерится тем же `render_prompt`, что и обучение: шаблон Qwen, инструменты, `enable_thinking=False`. Длина эталона — число токенов хвоста после этого промпта. Порог обучения для промпта — 3072 токена.

По этим длинам в экспериментах стоит `max_completion_length=256`: медиана эталона 68, p90 216, p99 435. Лимит 192 обрезает 13% эталонов, 256 — 6.4%.

In [ ]:
def percentile(values, fraction):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * fraction)]


tokenizer = load_tokenizer()
prompt_lengths = []
answer_lengths = []
for messages, answer, tools in train + validation:
    prompt = render_prompt(messages, tools, tokenizer)
    full = tokenizer.apply_chat_template(
        messages + [answer],
        tools=tools,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    if not full.startswith(prompt):
        raise ValueError("Промпт не является началом полного диалога")
    prompt_lengths.append(len(tokenizer(prompt, add_special_tokens=False)["input_ids"]))
    # tokenize=True возвращает BatchEncoding, у него len() == 2, а не число токенов.
    answer_lengths.append(len(tokenizer(full[len(prompt):], add_special_tokens=False)["input_ids"]))


def summarize(title, values):
    print(
        f"{title}: min {min(values)}, "
        f"p50 {percentile(values, 0.5)}, p90 {percentile(values, 0.9)}, "
        f"p99 {percentile(values, 0.99)}, max {max(values)}"
    )


summarize("промпт", prompt_lengths)
summarize("эталон", answer_lengths)
over_limit = sum(length > 3072 for length in prompt_lengths)
over_1024 = sum(length > 1024 for length in answer_lengths)
over_192 = sum(length > 192 for length in answer_lengths)
print(f"промптов длиннее 3072: {over_limit}")
print(f"эталонов длиннее 192: {over_192} ({over_192 / len(answer_lengths):.1%})")
print(f"эталонов длиннее 1024: {over_1024} ({over_1024 / len(answer_lengths):.1%})")

/home/edu/amsafin/HSE-Avito/LLM/hw5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


промпт: min 21, p50 719, p90 1249, p99 2016, max 4593
эталон: min 8, p50 68, p90 216, p99 435, max 1438
промптов длиннее 3072: 17
эталонов длиннее 192: 1708 (13.0%)
эталонов длиннее 1024: 3 (0.0%)


## Четыре разных эталона

По одному примеру каждого вида: первый ход с одним вызовом, несколько вызовов сразу, ход после результата инструмента, ответ обычным текстом.

In [ ]:
def last_user(messages):
    for message in reversed(messages):
        if message["role"] == "user":
            return message["content"]
    return ""


def show_example(title, messages, answer, tools):
    calls = answer.get("tool_calls") or []
    print("=" * 72)
    print(title)
    print(f"история {len(messages)} сообщений, инструментов {len(tools)}, вызовов {len(calls)}")
    print("последний вопрос:")
    print(last_user(messages)[:500])
    if calls:
        print("эталон:")
        for call in calls:
            function = call["function"]
            print(f"  {function['name']} {json.dumps(function['arguments'], ensure_ascii=False)[:300]}")
    else:
        print("эталон, текст:")
        print((answer.get("content") or "")[:500])


wanted = {
    "один вызов на первом ходе": lambda m, a, t: (
        not any(item["role"] == "assistant" for item in m) and len(a.get("tool_calls") or []) == 1
    ),
    "несколько вызовов в одном ответе": lambda m, a, t: len(a.get("tool_calls") or []) >= 2,
    "ход после результата инструмента": lambda m, a, t: any(item["role"] == "tool" for item in m) and a.get("tool_calls"),
    "ответ текстом": lambda m, a, t: not a.get("tool_calls"),
}
shown = set()
for messages, answer, tools in train:
    for title, match in wanted.items():
        if title not in shown and match(messages, answer, tools):
            show_example(title, messages, answer, tools)
            shown.add(title)
    if shown == set(wanted):
        break

один вызов на первом ходе
история 1 сообщений, инструментов 2, вызовов 1
последний вопрос:
As part of our conservation efforts, we're focusing on the cheetah habitat in the Savannah region. We need to evaluate the habitat health during spring and, based on that, track the movements of the Cheetah, Acinonyx jubatus, for the following 30 days. The evaluation criteria should include water quality, measured via spectral analysis, and soil nutrients, assessed through chemical testing. Could you help with that?
эталон:
  conservation.evaluate_habitat_health {"habitat_id": "Savannah region", "evaluation_criteria": [{"criterion": "water quality", "measurement_method": "spectral analysis"}, {"criterion": "soil nutrients", "measurement_method": "chemical testing"}]}
ход после результата инструмента
история 3 сообщений, инструментов 2, вызовов 1
последний вопрос:
As part of our conservation efforts, we're focusing on the cheetah habitat in the Savannah region. We need to evaluate the habitat heal